[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/07_frontier_alignment/07_frontier_alignment.ipynb)

# 07 · 前沿：CAI、self-play 与 weak-to-strong

> 《后训练与对齐》收尾模块 · 配套讲解：`07_讲解.html` · <span style="color:#888">CPU 可全程跑通；可选 OPENAI_API_KEY / 本地 Qwen2.5-0.5B 真跑 CAI 部分</span>

人类标注是 RLHF 流水线的瓶颈（成本/速度/专业性/一致性）。本 notebook 动手复现三条 **scalable oversight** 路线的最小可运行版本：

| 部分 | 内容 | 对应论文 |
|---|---|---|
| Part 1 | **CAI 玩具流水线**：constitution 驱动的 critique→revision 自我修订 | [Bai 2022] Constitutional AI, arXiv:2212.08073 |
| Part 2 | **Self-Rewarding 模拟**：policy 兼任 judge 的迭代 DPO；judge 偏差导致的奖励通胀崩塌曲线 | [Yuan 2024] Self-Rewarding LMs, arXiv:2401.10020 |
| Part 3 | **Weak-to-Strong 玩具实验**：弱老师监督强学生，复现 w2s generalization 并计算 PGR | [Burns 2023] Weak-to-Strong Generalization, arXiv:2312.09390 |

另见 [Guan 2024] Deliberative Alignment (arXiv:2412.16339)——其"显式推理安全政策"的思想在讲解 HTML 第 4 节展开，本 notebook 不单独建模。

**练习 3 道**：多原则修订管线 / PGR 计算 / 奖励通胀探测器。参考答案在文末。

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["axes.unicode_minus"] = False  # 兼容无中文字体环境，图内标签用英文

# ---- 可选真实 LLM 后端（缺省一律回退到确定性 mock，保证零依赖跑通）----
USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
QWEN_PIPE = None
# 想用本地小模型真跑 CAI（约 1GB 下载，CPU 数分钟）可手动取消注释：
# try:
#     from transformers import pipeline
#     QWEN_PIPE = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
# except Exception as e:
#     print("Qwen 加载失败，回退 mock:", e)

def real_llm_chat(prompt, max_new_tokens=200):
    '''统一的真实 LLM 入口；任何失败都返回 None（调用方回退 mock）。'''
    try:
        if USE_OPENAI:
            from openai import OpenAI
            client = OpenAI()
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_new_tokens, temperature=0)
            return r.choices[0].message.content
        if QWEN_PIPE is not None:
            out = QWEN_PIPE([{"role": "user", "content": prompt}],
                            max_new_tokens=max_new_tokens, do_sample=False)
            return out[0]["generated_text"][-1]["content"]
    except Exception as e:
        print("LLM 调用失败，回退 mock:", e)
    return None

print("后端:", "OpenAI API" if USE_OPENAI else ("Qwen2.5-0.5B" if QWEN_PIPE else "确定性 mock（默认）"))

## Part 1 · CAI 玩具流水线：constitution 驱动的自我修订

[Bai 2022] 的 **SL-CAI 阶段**：对一条有瑕疵的回答，反复执行 `critique(response, principle)` →
`revise(response, critique)`，得到修订后的回答作为 SFT 训练数据。人类只写**原则**，不写数据。

我们内嵌 3 条 constitution 原则和 4 条带瑕疵的回答。`critique` / `revise` 默认用**确定性规则版
mock LLM**（行为可预测、可断言），若上面检测到真实后端则自动换成真模型。

**程序化质量分**：因为瑕疵是规则可检测的（错误前提 / 无根据断言 / 填充语），我们能定义一个
规则可验证的代理质量分（proxy score），客观验证修订前后质量确实提升——这正是模块 06 强调的
"对齐声明必须可测量"。

In [ ]:
# ---- Constitution：3 条原则 ----
PRINCIPLES = {
    "premise":  "若回答建立在错误前提上，必须明确指出并纠正该前提，而不是顺着它说。",
    "evidence": "不做无根据的断言或猜测；证据不足时必须明确表达不确定性。",
    "concise":  "回答应简洁直接，删除无信息量的客套与填充语。",
}

# ---- 规则知识库：mock LLM 用它检测/修复瑕疵 ----
# 注意：纠正文本不得包含原错误说法的子串，否则会触发递归替换
FALSE_CLAIMS = {  # 错误说法 -> 纠正后的说法
    "长城是太空中唯一肉眼可见的人造建筑，所以宇航员都看得到":
        "在近地轨道上肉眼几乎无法看到长城，『太空唯一可见人造建筑』是流传甚广的误解",
    "人类只用了大脑的10%，剩下的部分都在沉睡":
        "『人类只用大脑10%』是已被神经科学否定的迷思，大脑各区域都会被使用",
}
SPECULATION_MARKERS = ["肯定是", "毫无疑问是", "我猜"]
FILLER_PHRASES = ["这是一个非常非常好的问题！", "首先请允许我说，", "总而言之，言而总之，", "众所周知，"]

# ---- 4 条带瑕疵的内嵌回答 ----
FLAWED = [
    {"q": "从太空能看到长城吗？",
     "a": "这是一个非常非常好的问题！长城是太空中唯一肉眼可见的人造建筑，所以宇航员都看得到。"},
    {"q": "他为什么辞职？",
     "a": "他肯定是和老板吵架了。"},
    {"q": "我们为什么只用一部分大脑？",
     "a": "众所周知，人类只用了大脑的10%，剩下的部分都在沉睡。"},
    {"q": "什么是过拟合？",
     "a": "首先请允许我说，过拟合是指模型把训练集中的噪声也记了下来，导致在新数据上表现变差。总而言之，言而总之，就是把答案背下来了。"},
]

def mock_critique(response, principle):
    '''确定性规则版 critique：违反原则 -> 返回带 [原则名] 标签的批评文本；否则 None。'''
    if principle == "premise":
        for claim in FALSE_CLAIMS:
            if claim in response:
                return f"[premise] 回答重复了错误前提：『{claim}』，应当指出并纠正。"
    elif principle == "evidence":
        if any(m in response for m in SPECULATION_MARKERS):
            return "[evidence] 回答包含无根据的断言或猜测，应改为明确表达不确定性。"
    elif principle == "concise":
        if any(p in response for p in FILLER_PHRASES):
            return "[concise] 回答包含无信息量的填充语，应删除。"
    return None

def mock_revise(response, critique):
    '''确定性规则版 revise：根据 critique 的 [标签] 决定修复动作。'''
    if critique is None:
        return response
    tag = critique.split("]")[0].lstrip("[")
    if tag == "premise":
        for claim, fix in FALSE_CLAIMS.items():
            response = response.replace(claim, fix)
    elif tag == "evidence":
        response = (response.replace("毫无疑问是", "可能是（证据有限，不能确定）")
                            .replace("肯定是", "可能是（证据有限，不能确定）")
                            .replace("我猜", "目前没有可靠信息，一种可能是"))
    elif tag == "concise":
        for p in FILLER_PHRASES:
            response = response.replace(p, "")
    return response.strip()

def llm_critique(response, principle):
    '''真实 LLM 版：失败时回退 mock。'''
    out = real_llm_chat(f"宪法原则：{PRINCIPLES[principle]}\n回答：{response}\n"
                        f"若回答违反该原则，用一句话指出问题（以 [{principle}] 开头）；若不违反，只输出 PASS。")
    if out is None:
        return mock_critique(response, principle)
    return None if "PASS" in out else out

def llm_revise(response, critique):
    if critique is None:
        return response
    out = real_llm_chat(f"原回答：{response}\n批评意见：{critique}\n请给出修订后的回答（只输出回答本身）。")
    return out if out is not None else mock_revise(response, critique)

# 默认管线使用 mock（确定性、可断言）；想用真模型把下面两行换成 llm_critique / llm_revise
critique_fn, revise_fn = mock_critique, mock_revise

def quality_score(response):
    '''规则可验证的代理质量分（满分 100）：错误前提 -35/处，无根据断言 -25，填充语 -10/处，过长 -1/10字。'''
    s = 100
    s -= 35 * sum(c in response for c in FALSE_CLAIMS)
    s -= 25 * int(any(m in response for m in SPECULATION_MARKERS))
    s -= 10 * sum(response.count(p) for p in FILLER_PHRASES)
    s -= max(0, len(response) - 120) // 10
    return max(s, 0)

print("constitution 原则数:", len(PRINCIPLES), "| 待修订回答数:", len(FLAWED))

In [ ]:
def cai_revise_loop(response, principles, critique_fn, revise_fn, max_rounds=3):
    '''SL-CAI 内循环：逐原则 critique→revision，直到一整轮无修改或达到 max_rounds。'''
    log = []
    for _ in range(max_rounds):
        changed = False
        for p in principles:
            c = critique_fn(response, p)
            if c is not None:
                response = revise_fn(response, c)
                log.append((p, c))
                changed = True
        if not changed:
            break
    return response, log

print(f"{'#':<3}{'修订前分':>8}{'修订后分':>8}  critique 次数")
results = []
for i, ex in enumerate(FLAWED):
    revised, log = cai_revise_loop(ex["a"], list(PRINCIPLES), critique_fn, revise_fn)
    s0, s1 = quality_score(ex["a"]), quality_score(revised)
    results.append((ex, revised, s0, s1, log))
    print(f"{i:<3}{s0:>8}{s1:>8}  {len(log)}")

print("\n=== 逐条前后对照 ===")
for i, (ex, revised, s0, s1, log) in enumerate(results):
    print(f"\n[{i}] Q: {ex['q']}")
    print(f"  修订前 ({s0}分): {ex['a']}")
    for p, c in log:
        print(f"    ↳ critique[{p}]: {c}")
    print(f"  修订后 ({s1}分): {revised}")

# 客观验证：每条都不降分，且总体严格提升；修订后不再触发任何 critique
assert all(s1 >= s0 for _, _, s0, s1, _ in results)
assert sum(r[3] for r in results) > sum(r[2] for r in results)
assert all(all(mock_critique(r[1], p) is None for p in PRINCIPLES) for r in results)
print("\n✅ CAI 自我修订使代理质量分全面提升，且修订后回答通过全部原则检查")
print("（在 [Bai 2022] 中，这些 (prompt, 修订后回答) 对就是 SL-CAI 的 SFT 数据）")

## Part 2 · Self-Rewarding 模拟：飞轮与崩塌曲线

[Yuan 2024] 的循环：policy 生成 N 个候选 → **同一个模型**当 judge 打分 → 取最高/最低分组成
DPO 偏好对 → 更新 policy → 下一轮。我们把它抽象成一个可控的数值模拟：

- policy 用两个参数刻画：平均**真实质量** `mu_q` 与平均**长度** `mu_len`；每轮从该分布采样候选。
- 回答过长会拖累真实质量（外部锚点能看到这一点）。
- judge 打分 = 真实质量 + **可控长度偏差** `judge_len_bias` + 噪声。
- "迭代 DPO" 的玩具版：policy 参数向被 judge 选中（chosen）样本的均值移动。

**核心教学点**：judge 无偏时，自评分与外部锚点分同步上升（飞轮正转）；judge 有长度偏差时，
自评分继续单调上涨（**奖励通胀**），外部锚点测到的真实质量却停滞→下跌（**偏好漂移**导致的崩塌）。
内部指标与外部指标脱钩，就是闭环失控的信号。

In [ ]:
def run_self_rewarding(n_iters=12, judge_len_bias=0.0, seed=0,
                       n_cand=400, top_frac=0.10, lr=0.6):
    '''Self-Rewarding 闭环的数值模拟。返回每轮的自评分(self)与外部锚点分(anchor)。'''
    rng = np.random.default_rng(seed)
    mu_q, mu_len = 0.0, 100.0
    L_GOOD = 120.0          # 超过该长度后，真实质量被冗长拖累
    LEN_PENALTY = 0.05      # 每超 1 个长度单位的质量损失
    hist = {"self": [], "anchor": [], "mu_len": []}
    for t in range(n_iters):
        base_q = rng.normal(mu_q, 1.0, n_cand)                  # 候选的内在质量
        length = np.maximum(rng.normal(mu_len, 25.0, n_cand), 5.0)
        true_q = base_q - LEN_PENALTY * np.maximum(length - L_GOOD, 0.0)  # 外部锚点看到的真实质量
        judge  = (true_q
                  + judge_len_bias * (length - 100.0) / 25.0   # judge 的长度偏差：无脑给长回答加分
                  + rng.normal(0.0, 0.3, n_cand))              # 打分噪声
        top = np.argsort(judge)[-int(top_frac * n_cand):]       # chosen 样本
        # 玩具版迭代 DPO：policy 分布向 chosen 的统计量移动
        mu_q   += lr * (base_q[top].mean() - mu_q)
        mu_len += lr * (length[top].mean() - mu_len)
        hist["self"].append(judge[top].mean())    # 内部指标：自评奖励
        hist["anchor"].append(true_q.mean())      # 外部指标：独立锚点测到的真实质量
        hist["mu_len"].append(mu_len)
    return hist

hist_ok  = run_self_rewarding(judge_len_bias=0.0, seed=SEED)   # judge 无偏
hist_bad = run_self_rewarding(judge_len_bias=2.5, seed=SEED)   # judge 偏爱长回答

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, hist, title in [(axes[0], hist_ok, "Unbiased judge (flywheel works)"),
                        (axes[1], hist_bad, "Length-biased judge (reward inflation)")]:
    ax.plot(hist["self"], "o-", label="self-reward (internal)")
    ax.plot(hist["anchor"], "s--", label="external anchor (true quality)")
    ax.set_title(title); ax.set_xlabel("iteration"); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel("score")
plt.tight_layout(); plt.show()

print(f"无偏 judge : 自评 {hist_ok['self'][0]:+.2f}→{hist_ok['self'][-1]:+.2f} | "
      f"锚点 {hist_ok['anchor'][0]:+.2f}→{hist_ok['anchor'][-1]:+.2f} | 平均长度 {hist_ok['mu_len'][-1]:.0f}")
print(f"长度偏差 judge: 自评 {hist_bad['self'][0]:+.2f}→{hist_bad['self'][-1]:+.2f} | "
      f"锚点 {hist_bad['anchor'][0]:+.2f}→{hist_bad['anchor'][-1]:+.2f} | 平均长度 {hist_bad['mu_len'][-1]:.0f}")

assert hist_ok["anchor"][-1] > hist_ok["anchor"][0] + 1.0      # 无偏：真实质量显著上升
assert hist_bad["self"][-1] > hist_bad["self"][0] + 1.0        # 有偏：自评奖励照样通胀
assert hist_bad["anchor"][-1] < hist_bad["anchor"][0]          # 有偏：真实质量反而下降 = 崩塌
print("\n✅ 复现核心现象：judge 有偏时『自评分上涨 + 真实质量下跌』——没有外部锚点就发现不了")

## Part 3 · Weak-to-Strong 玩具实验（纯 numpy，全程 CPU 秒级）

[Burns 2023] 的范式平移到一个合成二分类任务上：

1. **真实任务**：10 维特征，真标签由一个线性边界决定（强模型的假设类能精确表达它）。
2. **weak teacher**：只见过 **30 条金标**的 1-NN 分类器——容量受限、边界粗糙，相当于"弱监督者"。
3. weak teacher 给 4000 条无标数据打**弱标签**（带错误）。
4. **strong student**：逻辑回归（假设类与真实边界同构 = "强模型的预训练表征"），**只**在弱标签上训练。
5. **strong ceiling**：同样的逻辑回归直接在金标上训练（强模型潜力上限）。

预期：student 测试准确率**超过**它的老师——老师的错误在 student 的（线性）假设空间里近似随机噪声，
被大样本拟合平均掉了；这就是 **weak-to-strong generalization**。再算
$\mathrm{PGR} = \frac{acc_{w2s} - acc_{weak}}{acc_{ceiling} - acc_{weak}}$。

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def train_logreg(X, y, lr=0.5, iters=600, l2=1e-3):
    '''极简逻辑回归（全批量 GD），y ∈ {0,1}。'''
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(iters):
        g = sigmoid(X @ w + b) - y
        w -= lr * (X.T @ g / len(y) + l2 * w)
        b -= lr * g.mean()
    return w, b

def logreg_acc(w, b, X, y):
    return float(((X @ w + b > 0).astype(int) == y).mean())

def knn1_predict(X_train, y_train, X):
    '''1-NN：weak teacher（容量受限的弱模型）。'''
    d2 = ((X[:, None, :] - X_train[None, :, :]) ** 2).sum(-1)
    return y_train[d2.argmin(1)]

# ---- 合成任务 ----
rng = np.random.default_rng(SEED)
D = 10
w_true = rng.normal(size=D)
def make(n):
    X = rng.normal(size=(n, D))
    return X, (X @ w_true > 0).astype(int)

X_gold, y_gold = make(30)       # 仅 30 条金标 -> weak teacher 的全部知识
X_pool, _      = make(4000)     # 大量无标数据（弱老师来打标）
X_test, y_test = make(2000)

# ① weak teacher（1-NN on 30 gold）
acc_weak = float((knn1_predict(X_gold, y_gold, X_test) == y_test).mean())
# ② 弱标签
y_weak_pool = knn1_predict(X_gold, y_gold, X_pool)
teacher_label_err = float((y_weak_pool != (X_pool @ w_true > 0).astype(int)).mean())
# ③ strong student：只看弱标签
w_s, b_s = train_logreg(X_pool, y_weak_pool)
acc_w2s = logreg_acc(w_s, b_s, X_test, y_test)
# ④ strong ceiling：同样的学生直接看金标
w_c, b_c = train_logreg(X_pool, (X_pool @ w_true > 0).astype(int))
acc_ceiling = logreg_acc(w_c, b_c, X_test, y_test)

pgr_val = (acc_w2s - acc_weak) / (acc_ceiling - acc_weak)
print(f"weak teacher (1-NN, 30 金标)      test acc = {acc_weak:.3f}")
print(f"  └ 它打的弱标签错误率              = {teacher_label_err:.3f}")
print(f"strong student (只看弱标签)        test acc = {acc_w2s:.3f}")
print(f"strong ceiling (直接看金标)        test acc = {acc_ceiling:.3f}")
print(f"Performance Gap Recovered (PGR)        = {pgr_val:.2f}")

bars = ["weak\nteacher", "strong on\nweak labels", "strong ceiling\non gold"]
vals = [acc_weak, acc_w2s, acc_ceiling]
plt.figure(figsize=(6, 4))
plt.bar(bars, vals, color=["#888", "#4878cf", "#2ca02c"])
for i, v in enumerate(vals):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.ylim(0.5, 1.02); plt.ylabel("test accuracy")
plt.title(f"Weak-to-Strong Generalization (PGR = {pgr_val:.2f})")
plt.tight_layout(); plt.show()

assert acc_w2s > acc_weak + 0.03, "strong 学生应明显超过 weak 老师"
assert acc_ceiling > acc_w2s, "弱监督仍然损失了一部分潜力（PGR < 1）"
print("\n✅ 复现 weak-to-strong generalization：学生在『全部由老师标注』的数据上训练，却超过了老师")
print("   机制：老师的错误在学生的线性假设空间里近似噪声，被大样本平均掉；学生拟合到的是信号")

## ✏️ 练习 1：多原则顺序修订管线 `apply_critiques`

把 Part 1 的内循环封装成通用函数：

```
apply_critiques(response, principles, critique_fn, revise_fn, max_rounds=3)
  -> (final_response, log)
```

要求：
1. 每一轮按 `principles` 的顺序逐条调用 `critique_fn(response, p)`；返回非 `None` 时调用
   `revise_fn(response, critique)` 更新 response，并把 `(p, critique)` 追加进 `log`；
2. 若**完整一轮**没有任何原则触发修订，提前停止；最多跑 `max_rounds` 轮；
3. 干净的回答应原样返回且 `log == []`。

提示：10 行左右；注意"提前停止"的判断要放在一整轮结束后。

In [ ]:
def apply_critiques(response, principles, critique_fn, revise_fn, max_rounds=3):
    '''多原则顺序 critique→revision 管线。返回 (final_response, log)。'''
    log = []
    # TODO: 外层最多 max_rounds 轮；内层按顺序遍历 principles，
    #       critique 非 None 则 revise 并记录；一轮无修改则 break
    raise NotImplementedError
    return response, log

In [ ]:
# ---- 练习 1 自测 ----
r1, log1 = apply_critiques(FLAWED[0]["a"], list(PRINCIPLES), mock_critique, mock_revise)
assert all(mock_critique(r1, p) is None for p in PRINCIPLES), "修订后不应再触发任何原则"
assert len(log1) >= 2, "该回答同时违反 premise 与 concise，log 至少 2 条"
assert quality_score(r1) > quality_score(FLAWED[0]["a"]), "代理质量分应提升"

clean = "在近地轨道上肉眼几乎无法看到长城。"
r2, log2 = apply_critiques(clean, list(PRINCIPLES), mock_critique, mock_revise)
assert r2 == clean and log2 == [], "干净回答应原样返回且 log 为空"

# 边界：max_rounds=1 时也只跑一轮就返回
_, log3 = apply_critiques(FLAWED[1]["a"], list(PRINCIPLES), mock_critique, mock_revise, max_rounds=1)
assert all(p in PRINCIPLES for p, _ in log3)
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `pgr(weak_acc, w2s_acc, strong_ceiling_acc)`

实现 [Burns 2023] 的 Performance Gap Recovered：

$$\mathrm{PGR} = \frac{acc_{w2s} - acc_{weak}}{acc_{ceiling} - acc_{weak}}$$

要求：
1. 正常情况返回该比值（float）；
2. 当 `strong_ceiling_acc <= weak_acc` 时（gap 不存在，指标无意义）`raise ValueError`；
3. 边界语义：学生只会复刻老师 → 0；弱监督完全不损失潜力 → 1。

提示：3–5 行。

In [ ]:
def pgr(weak_acc, w2s_acc, strong_ceiling_acc):
    '''Performance Gap Recovered, [Burns 2023]。'''
    # TODO: 先处理 strong_ceiling_acc <= weak_acc 的非法情形，再返回比值
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
assert abs(pgr(0.70, 0.70, 0.90) - 0.0) < 1e-12, "w2s == weak 时 PGR = 0"
assert abs(pgr(0.70, 0.90, 0.90) - 1.0) < 1e-12, "w2s == ceiling 时 PGR = 1"
assert abs(pgr(0.70, 0.80, 0.90) - 0.5) < 1e-12
assert pgr(0.70, 0.65, 0.90) < 0, "学生不如老师 -> PGR 为负（imitation failure 的信号）"
try:
    pgr(0.90, 0.80, 0.90)
    raise AssertionError("ceiling <= weak 应抛 ValueError")
except ValueError:
    pass
# 用 Part 3 的真实实验数据交叉验证
assert abs(pgr(acc_weak, acc_w2s, acc_ceiling) - pgr_val) < 1e-9
print("✅ 练习 2 通过")

## ✏️ 练习 3：奖励通胀探测器 `reward_inflation_detector`

Part 2 的教训：**自评分数（internal）与外部锚点分数（external anchor）的趋势脱钩 = 通胀警报**。
实现：

```
reward_inflation_detector(self_scores, anchor_scores, threshold=0.5) -> bool
```

要求：
1. 分别对两条序列拟合线性趋势斜率（提示：`np.polyfit(np.arange(n), scores, 1)[0]`）；
2. 当 `slope(self) - slope(anchor) > threshold` 时返回 `True`（报警），否则 `False`；
3. 两条序列健康同涨、或同样平稳时不应报警。

提示：4–6 行。这正是讲解 HTML 第 7 节"冻结外部锚 + 差值漂移监控"的最小实现。

In [ ]:
def reward_inflation_detector(self_scores, anchor_scores, threshold=0.5):
    '''自评分数相对外部锚点的趋势漂移检测。漂移超过 threshold 返回 True。'''
    # TODO: 用 np.polyfit 取两条序列的斜率，比较斜率差与 threshold
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
healthy_self   = [1.0, 2.0, 3.0, 4.0, 5.0]
healthy_anchor = [0.9, 1.8, 3.1, 3.9, 5.2]
assert reward_inflation_detector(healthy_self, healthy_anchor) is False, "健康同涨不报警"

flat = [1.0, 1.1, 0.9, 1.0, 1.05]
assert reward_inflation_detector(flat, flat) is False, "同样平稳不报警"

inflated_self   = [1.0, 2.0, 3.2, 4.5, 6.0]     # 自评一路上涨
inflated_anchor = [1.0, 1.1, 0.8, 0.4, 0.0]     # 真实质量停滞下跌
assert reward_inflation_detector(inflated_self, inflated_anchor) is True, "注入通胀必须报警"

# 在 Part 2 的两次模拟上验证：无偏不报警，有偏报警
assert reward_inflation_detector(hist_ok["self"],  hist_ok["anchor"])  is False
assert reward_inflation_detector(hist_bad["self"], hist_bad["anchor"]) is True
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。每题一个独立 cell。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def apply_critiques(response, principles, critique_fn, revise_fn, max_rounds=3):
    log = []
    for _ in range(max_rounds):
        changed = False
        for p in principles:
            c = critique_fn(response, p)
            if c is not None:
                response = revise_fn(response, c)
                log.append((p, c))
                changed = True
        if not changed:
            break
    return response, log

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def pgr(weak_acc, w2s_acc, strong_ceiling_acc):
    if strong_ceiling_acc <= weak_acc:
        raise ValueError("strong ceiling 必须高于 weak teacher，否则 PGR 无意义")
    return (w2s_acc - weak_acc) / (strong_ceiling_acc - weak_acc)

In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def reward_inflation_detector(self_scores, anchor_scores, threshold=0.5):
    t = np.arange(len(self_scores))
    slope_self   = np.polyfit(t, np.asarray(self_scores, float), 1)[0]
    slope_anchor = np.polyfit(np.arange(len(anchor_scores)), np.asarray(anchor_scores, float), 1)[0]
    return bool(slope_self - slope_anchor > threshold)

## 🧭 全课总结：《后训练与对齐》8 模块一句话脉络

| 模块 | 一句话 |
|---|---|
| 00 · Setup | 环境与全课地图：后训练 = 把"会接龙的基座"变成"对齐的助手" |
| 01 · SFT 与 loss masking | 只对 response token 算 loss，用少量高质量示范教会模型"做助手"这个格式 [Ouyang 2022; Zhou 2023] |
| 02 · 奖励模型 | Bradley–Terry 把成对偏好压成标量奖励；proxy 奖励可被过度优化 [Gao 2022] |
| 03 · RLHF 与 PPO | 在 KL 约束下最大化奖励：既要讨好 RM，又不能漂离参考策略 [Schulman 2017] |
| 04 · DPO 家族 | 跳过显式 RM 与 RL，把偏好优化写成闭式分类损失 [Rafailov 2023] |
| 05 · RLVR 与 GRPO | 可程序化验证的奖励 + 组相对优势，催生推理模型 [Shao 2024; DeepSeek-AI 2025] |
| 06 · 对齐的评测 | win-rate、judge 偏差、对齐税——对齐声明必须可测量 [Zheng 2023; Dubois 2024] |
| 07 · 前沿（本章） | 人类标注到顶后：CAI 用宪法造数据 [Bai 2022]，self-rewarding 内化裁判 [Yuan 2024]，deliberative alignment 用推理执行政策 [Guan 2024]，weak-to-strong 直面"弱监督强" [Burns 2023] |

贯穿八个模块的同一根主线：**监督信号从哪里来、它有什么系统性缺陷、policy 会如何利用这些缺陷、
以及评测如何独立地发现这一切。**

🎉 **恭喜完成《后训练与对齐》全部 8 个模块！** 返回 [课程主页](../index.html) 查看全课地图、
[术语词典](../glossary.md) 与 [论文清单](../references.md)。

---
## 🎯 真实数据胶囊题：真实 GSM8K 上的 self-consistency（多数投票）

前沿自改进的一个基石：对同一题多次采样、取**多数答案**，比单次更准。用真实 GSM8K 金标，模拟带噪采样，验证多数投票的准确率高于单次采样。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

rows=gsm8k(100)
def gold(a): return a.split('####')[-1].strip().replace(',','')
golds=[gold(r["answer"]) for r in rows]
rng=np.random.default_rng(0)
def sample_answers(g, k=5, p_correct=0.55):
    # 模拟：每次以 p_correct 给对答案，否则给一个错答案
    out=[]
    for _ in range(k):
        out.append(g if rng.random()<p_correct else str(int(rng.integers(0,99999))))
    return out
print(f"{len(rows)} 道真实题, 单次采样正确率设为 0.55")

**练习**：实现 `majority_vote(answers)`（返回出现最多的答案）。用它统计：单次 vs 5 次多数投票的准确率。验证多数投票更高。

In [ ]:
def majority_vote(answers):
    # TODO: 返回 answers 里出现次数最多的元素
    raise NotImplementedError


In [ ]:
# 自测
assert majority_vote(["7","7","3"])=="7"
single=0; voted=0
for g in golds:
    samples=sample_answers(g, k=5, p_correct=0.55)
    single += (samples[0]==g)
    voted  += (majority_vote(samples)==g)
single/=len(golds); voted/=len(golds)
assert voted > single, "多数投票应优于单次"
print(f"self-consistency ✓  单次 {single:.2f} -> 5次多数投票 {voted:.2f}")


### 📖 参考答案

In [ ]:
def majority_vote(answers):
    from collections import Counter
    return Counter(answers).most_common(1)[0][0]
print("✓ self-consistency: 多条推理路径投票，是测试时计算(C9)与自改进的基础")